# 03 — Segmentação de substâncias por perfil de mercado (clustering)

**Pergunta:** existem "tipos de mercado" distintos entre os medicamentos vendidos no Brasil?

Cada **substância** vira uma linha, descrita por características do seu mercado:
preço, número de concorrentes e participação de genéricos e de produtos novos/biológicos.
Usamos **K-Means** para agrupar substâncias com perfis parecidos e depois interpretamos cada grupo.

**Base:** `cmed_analise_*.csv`, gerado pelo `02_eda.ipynb`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")

SEMENTE = 42
PASTA_PROCESSED = Path("../data/processed")
PASTA_FIGURAS = Path("../reports/figures")
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

def salvar(fig, nome):
    fig.savefig(PASTA_FIGURAS / f"{nome}.png", dpi=150, bbox_inches="tight")

ARQUIVO = sorted(PASTA_PROCESSED.glob("cmed_analise_*.csv"))[-1]
df = pd.read_csv(ARQUIVO)
print(ARQUIVO.name, df.shape)

## 1. Uma linha por substância

| Variável | O que mede |
|---|---|
| `pmc_mediano` | nível de preço típico da substância |
| `laboratorios` | quantos fabricantes vendem a substância (concorrência) |
| `pct_generico` | fração das apresentações que são genéricos |
| `pct_novo_bio` | fração que são medicamentos novos ou biológicos (inovação/proteção) |

Outras variáveis (área terapêutica, se é associação de princípios ativos) ficam de fora do modelo
e são usadas depois, só para interpretar os grupos.

In [ ]:
por_substancia = df.groupby("substancia")

base = pd.DataFrame({
    "pmc_mediano": por_substancia["pmc_sem_impostos"].median(),
    "laboratorios": por_substancia["laboratorio"].nunique(),
    "apresentacoes": por_substancia.size(),
    "pct_generico": por_substancia["tipo_produto"].agg(lambda s: (s == "Genérico").mean()),
    "pct_novo_bio": por_substancia["tipo_produto"].agg(lambda s: s.isin(["Novo", "Biológico"]).mean()),
    "pct_biologico": por_substancia["tipo_produto"].agg(lambda s: (s == "Biológico").mean()),
    "associacao": por_substancia["n_substancias"].first() > 1,
    "area": por_substancia["area"].agg(lambda s: s.mode().iat[0]),
})

print(f"{len(base):,} substâncias")
base.describe().T

## 2. Preparação das variáveis

Preço e número de laboratórios são muito assimétricos: poucas substâncias concentram valores
enormes. O K-Means usa distâncias, então esses extremos dominariam o resultado.
Aplicamos **logaritmo** nessas duas variáveis e depois **padronizamos** todas
(média 0, desvio 1) para que tenham o mesmo peso.

In [ ]:
X = pd.DataFrame({
    "log_preco": np.log10(base["pmc_mediano"]),
    "log_laboratorios": np.log1p(base["laboratorios"]),
    "pct_generico": base["pct_generico"],
    "pct_novo_bio": base["pct_novo_bio"],
}, index=base.index)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, col in zip(axes, X.columns):
    sns.histplot(X[col], bins=30, ax=ax, color="#264653")
    ax.set_title(col)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

scaler = StandardScaler()
X_pad = scaler.fit_transform(X)

## 3. Escolha do número de grupos

Testamos de 2 a 8 grupos e olhamos duas métricas:
- **Inércia (método do cotovelo):** soma das distâncias ao centro do grupo. Procuramos o ponto
  em que adicionar grupos deixa de reduzir muito essa soma.
- **Silhueta:** mede o quanto cada ponto está mais perto do próprio grupo do que dos outros
  (de -1 a 1, quanto maior, melhor).

In [ ]:
resultados = []
for k in range(2, 9):
    modelo = KMeans(n_clusters=k, n_init=20, random_state=SEMENTE).fit(X_pad)
    resultados.append({
        "k": k,
        "inercia": modelo.inertia_,
        "silhueta": silhouette_score(X_pad, modelo.labels_),
    })
resultados = pd.DataFrame(resultados).set_index("k")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(resultados.index, resultados["inercia"], marker="o", color="#264653")
axes[0].set_title("Método do cotovelo")
axes[0].set_xlabel("Número de grupos (k)")
axes[0].set_ylabel("Inércia")
axes[1].plot(resultados.index, resultados["silhueta"], marker="o", color="#2a9d8f")
axes[1].set_title("Silhueta")
axes[1].set_xlabel("Número de grupos (k)")
for ax in axes:
    ax.axvline(4, color="#e76f51", ls="--")
plt.tight_layout()
salvar(fig, "06_escolha_k")
plt.show()

resultados

A queda da inércia desacelera a partir de **k = 4**, e a silhueta dá um salto até esse ponto,
com ganhos pequenos depois. Com 5 grupos a silhueta é um pouco maior, mas o grupo extra
apenas divide o mercado competitivo em duas partes parecidas.
Escolhemos **k = 4**, que equilibra qualidade estatística e facilidade de interpretação.

In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, n_init=20, random_state=SEMENTE).fit(X_pad)
base["cluster"] = kmeans.labels_

centros = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=X.columns)
centros

## 4. Dando nome aos grupos

Os números dos clusters são arbitrários (mudam se o modelo for treinado de novo).
Por isso os nomes são atribuídos por **regras sobre os centros**, e não fixados no número:

1. maior preço → **Alto custo e especialidade**
2. mais laboratórios → **Mercado competitivo**
3. dos dois restantes, maior participação de novos/biológicos → **Novos sem genérico**
4. o último → **Similares e marcas populares**

In [ ]:
restantes = list(centros.index)
nomes = {}

alto_custo = centros.loc[restantes, "log_preco"].idxmax()
nomes[alto_custo] = "Alto custo e especialidade"
restantes.remove(alto_custo)

competitivo = centros.loc[restantes, "log_laboratorios"].idxmax()
nomes[competitivo] = "Mercado competitivo"
restantes.remove(competitivo)

marca = centros.loc[restantes, "pct_novo_bio"].idxmax()
nomes[marca] = "Novos sem genérico"
restantes.remove(marca)

nomes[restantes[0]] = "Similares e marcas populares"

ORDEM = ["Mercado competitivo", "Similares e marcas populares",
         "Novos sem genérico", "Alto custo e especialidade"]
CORES = dict(zip(ORDEM, ["#2a9d8f", "#e9c46a", "#f4a261", "#6d597a"]))

base["segmento"] = pd.Categorical(base["cluster"].map(nomes), categories=ORDEM, ordered=True)
base["segmento"].value_counts().sort_index()

## 5. Perfil de cada segmento

In [ ]:
perfil = base.groupby("segmento", observed=True).agg(
    substancias=("pmc_mediano", "size"),
    pmc_mediano=("pmc_mediano", "median"),
    laboratorios_mediana=("laboratorios", "median"),
    pct_generico=("pct_generico", "mean"),
    pct_novo_bio=("pct_novo_bio", "mean"),
    pct_biologico=("pct_biologico", "mean"),
    pct_associacoes=("associacao", "mean"),
)
perfil.style.format({
    "substancias": "{:,.0f}", "pmc_mediano": "R$ {:,.2f}", "laboratorios_mediana": "{:.0f}",
    "pct_generico": "{:.0%}", "pct_novo_bio": "{:.0%}", "pct_biologico": "{:.0%}", "pct_associacoes": "{:.0%}",
})

In [ ]:
# Mapa de calor: quanto cada segmento está acima (+) ou abaixo (-) da média em cada variável
centros_pad = pd.DataFrame(kmeans.cluster_centers_, columns=X.columns)
centros_pad.index = centros_pad.index.map(nomes)
centros_pad = centros_pad.loc[ORDEM]
centros_pad.columns = ["Preço (log)", "Laboratórios (log)", "% genéricos", "% novos/biológicos"]

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(centros_pad, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax,
            cbar_kws={"label": "desvios em relação à média"})
ax.set_title("Perfil dos segmentos (variáveis padronizadas)")
ax.set_ylabel("")
salvar(fig, "07_perfil_segmentos")
plt.show()

In [ ]:
# Projeção em 2 dimensões com PCA, só para visualização
pca = PCA(n_components=2, random_state=SEMENTE)
coords = pca.fit_transform(X_pad)
base["pc1"], base["pc2"] = coords[:, 0], coords[:, 1]
print(f"Variância explicada pelos 2 componentes: {pca.explained_variance_ratio_.sum():.0%}")

cargas = pd.DataFrame(pca.components_.T, index=X.columns, columns=["PC1", "PC2"])
display(cargas)

fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(data=base, x="pc1", y="pc2", hue="segmento", palette=CORES,
                s=25, alpha=0.7, ax=ax)

destaques = ["ADALIMUMABE", "PARACETAMOL", "ROSUVASTATINA CÁLCICA", "IBUPROFENO", "SOMATROPINA"]
for nome in destaques:
    if nome in base.index:
        linha = base.loc[nome]
        ax.annotate(nome.title(), (linha["pc1"], linha["pc2"]), fontsize=9,
                    xytext=(5, 5), textcoords="offset points")

ax.set_title("Substâncias por segmento (projeção PCA)")
ax.set_xlabel("Componente 1")
ax.set_ylabel("Componente 2")
ax.legend(title="")
salvar(fig, "08_segmentos_pca")
plt.show()

O PCA serve apenas para visualizar os grupos em duas dimensões; o modelo foi treinado com as quatro variáveis.
A tabela de cargas mostra o que cada eixo representa (valores altos = variável pesa muito naquele eixo).

As **linhas diagonais** no gráfico são substâncias com um único laboratório e 0% ou 100% de produtos novos:
para elas, só o preço varia, então os pontos se alinham. Na linha da direita (100% novos), o preço é
o que separa *Novos sem genérico* de *Alto custo e especialidade*.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

tipos = (df.merge(base[["segmento"]], left_on="substancia", right_index=True)
           .pipe(lambda d: pd.crosstab(d["segmento"], d["tipo_produto"], normalize="index")))
tipos = tipos[["Genérico", "Similar", "Novo", "Biológico", "Específico", "Fitoterápico"]]
tipos.plot(kind="barh", stacked=True, ax=axes[0], width=0.7,
           color=["#2a9d8f", "#e9c46a", "#e76f51", "#6d597a", "#8d99ae", "#90be6d"])
axes[0].set_title("Tipos de produto em cada segmento")
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].set_ylabel("")
axes[0].legend(bbox_to_anchor=(1, 1), fontsize=8)
axes[0].invert_yaxis()

areas_top = base["area"].value_counts().head(6).index
areas = pd.crosstab(base["segmento"], base["area"].where(base["area"].isin(areas_top), "Outras"),
                    normalize="index")
areas.plot(kind="barh", stacked=True, ax=axes[1], width=0.7, colormap="tab20")
axes[1].set_title("Áreas terapêuticas em cada segmento")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_ylabel("")
axes[1].legend(bbox_to_anchor=(1, 1), fontsize=8)
axes[1].invert_yaxis()

plt.tight_layout()
salvar(fig, "09_composicao_segmentos")
plt.show()

## 6. Exemplos de cada segmento

In [ ]:
for segmento in ORDEM:
    exemplos = (base[base["segmento"] == segmento]
                .sort_values(["laboratorios", "apresentacoes"], ascending=False)
                .head(8))
    print(f"\n=== {segmento} ===")
    print(exemplos[["laboratorios", "pmc_mediano", "pct_generico", "area"]].to_string())

## 7. Leitura dos segmentos

- **Mercado competitivo:** remédios de uso comum (paracetamol, ibuprofeno, rosuvastatina),
  com vários fabricantes e forte presença de genéricos. Preço baixo e, como vimos no `02_eda`,
  grande dispersão de preço entre laboratórios.
- **Similares e marcas populares:** em geral um único fabricante por substância, e quase nenhum produto novo. São marcas de
  similares, fitoterápicos, vitaminas e associações em dose fixa. Preço baixo sem depender de genéricos.
- **Novos sem genérico:** medicamentos novos, em geral de um só fabricante e com preço moderado.
  Quase metade são associações em dose fixa, e praticamente nenhum tem versão genérica. São candidatos naturais
  a sofrer concorrência de genéricos no futuro.
- **Alto custo e especialidade:** biológicos, oncológicos e imunológicos (adalimumabe, somatropina),
  com preço mediano na casa dos milhares de reais e praticamente nenhum concorrente.

**Uso prático:** um laboratório de genéricos pode olhar o segmento *Novos sem genérico*
para priorizar oportunidades de lançamento; um órgão de compras públicas pode focar negociação
no segmento *Alto custo e especialidade*, onde estão os preços mais altos e quase não há concorrência.

**Limitações:**
- K-Means assume grupos aproximadamente esféricos; as fronteiras entre segmentos são graduais, não rígidas.
- Muitas substâncias têm um único laboratório, o que concentra pontos nas mesmas regiões do espaço.
- Os segmentos refletem o **catálogo** (apresentações registradas), não o volume de vendas.

In [ ]:
data_str = ARQUIVO.stem.split("_")[-1]
saida = PASTA_PROCESSED / f"cmed_segmentos_{data_str}.csv"
base.drop(columns=["cluster", "pc1", "pc2"]).to_csv(saida, encoding="utf-8")
print("Salvo em", saida)